In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ==========================================================
# Handwritten English Character Recognition using CNN
# ==========================================================


# Import Libraries

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

warnings.filterwarnings("ignore")

print("TensorFlow Version :", tf.__version__)

In [ ]:
# ==========================================================
# Check GPU
# ==========================================================

gpus = tf.config.list_physical_devices('GPU')

if len(gpus) > 0:
    print("GPU is Available")
    print(gpus)
else:
    print("GPU is NOT Available")

In [4]:
# ==========================================================
# Dataset Path
# ==========================================================

train_path = "/kaggle/input/datasets/rahulkumar1822/character-dataset/combined_folder/train"

test_path = "/kaggle/input/datasets/rahulkumar1822/character-dataset/combined_folder/test"

In [ ]:
# ==========================================================
# Verify Dataset
# ==========================================================

print("Train Exists :", os.path.exists(train_path))
print("Test Exists :", os.path.exists(test_path))

In [ ]:
# ==========================================================
# Check Number of Classes
# ==========================================================

print("Train Classes :", len(os.listdir(train_path)))
print("Test Classes  :", len(os.listdir(test_path)))

In [ ]:
import os

classes_name = sorted([
    folder
    for folder in os.listdir(train_path)
    if os.path.isdir(os.path.join(train_path, folder))
])

NUM_CLASSES = len(classes_name)

print("Number of Classes:", NUM_CLASSES)

In [ ]:
# ==========================================================
# Count Images in Each Class
# ==========================================================

image_count = {}

for cls in classes_name:

    folder = os.path.join(train_path, cls)

    images = [
        img
        for img in os.listdir(folder)
        if img.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    image_count[cls] = len(images)

dataset_info = pd.DataFrame({
    "Class": list(image_count.keys()),
    "Images": list(image_count.values())
})

dataset_info

In [ ]:
# ==========================================================
# Dataset Statistics
# ==========================================================

print("Total Images   :", dataset_info["Images"].sum())

print("Average Images :", dataset_info["Images"].mean())

print("Maximum Images :", dataset_info["Images"].max())

print("Minimum Images :", dataset_info["Images"].min())

In [ ]:
# ==========================================================
# Plot Class Distribution
# ==========================================================

plt.figure(figsize=(18,6))

plt.bar(dataset_info["Class"], dataset_info["Images"])

plt.xlabel("Classes")

plt.ylabel("Number of Images")

plt.title("Training Images per Class")

plt.xticks(rotation=90)

plt.show()

In [11]:
# ==========================================================
# Dataset Parameters
# ==========================================================

IMAGE_SIZE = (28,28)

BATCH_SIZE = 32

SEED = 42

In [ ]:
# ==========================================================
# Load Training Dataset
# ==========================================================

train_dataset = tf.keras.utils.image_dataset_from_directory(

    directory=train_path,

    labels="inferred",

    label_mode="int",

    validation_split=0.20,

    subset="training",

    seed=SEED,

    image_size=IMAGE_SIZE,

    color_mode="grayscale",

    batch_size=BATCH_SIZE,

    shuffle=True

)

In [ ]:
# ==========================================================
# Validation Dataset
# ==========================================================

validation_dataset = tf.keras.utils.image_dataset_from_directory(

    directory=train_path,

    labels="inferred",

    label_mode="int",

    validation_split=0.20,

    subset="validation",

    seed=SEED,

    image_size=IMAGE_SIZE,

    color_mode="grayscale",

    batch_size=BATCH_SIZE,

    shuffle=True

)

In [ ]:
# ==========================================================
# Load Test Dataset
# ==========================================================

test_dataset = tf.keras.utils.image_dataset_from_directory(

    directory=test_path,

    labels="inferred",

    label_mode="int",

    image_size=IMAGE_SIZE,

    color_mode="grayscale",

    batch_size=BATCH_SIZE,

    shuffle=False

)

In [ ]:
# ==========================================================
# Dataset Summary
# ==========================================================

print("Training Batches   :", len(train_dataset))
print("Validation Batches :", len(validation_dataset))
print("Testing Batches    :", len(test_dataset))

print("Number of Classes  :", len(classes_name))

In [ ]:
# ==========================================================
# Display Sample Images
# ==========================================================

plt.figure(figsize=(10,10))

for images, labels in train_dataset.take(1):

    for i in range(min(16, len(images))):

        plt.subplot(4,4,i+1)

        plt.imshow(images[i].numpy().squeeze(), cmap="gray")

        plt.title(classes_name[int(labels[i])])

        plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# ==========================================================
# Image Shape
# ==========================================================

for images, labels in train_dataset.take(1):

    print("Images Shape :", images.shape)

    print("Labels Shape :", labels.shape)

In [ ]:
#================================================
#To verify images come from different folders
#================================================

for images, labels in train_dataset.take(1):

    print("Labels:")
    print(labels.numpy())

In [ ]:
#=====================================
# print the corresponding class names
#=====================================

for images, labels in train_dataset.take(1):

    print("Classes in this batch:")

    for label in labels.numpy():
        print(classes_name[label])

In [ ]:
# ==========================================================
# Pixel Value Range
# ==========================================================

for images, labels in train_dataset.take(1):

    print("Minimum Pixel :", tf.reduce_min(images).numpy())

    print("Maximum Pixel :", tf.reduce_max(images).numpy())

In [21]:
# ==========================================================
# Normalize Images
# ==========================================================

# Normalize pixel values from [0,255] to [0,1]
normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)

AUTOTUNE = tf.data.AUTOTUNE

# Training Dataset

train_dataset = train_dataset.map(
    lambda images, labels: (normalization_layer(images), labels),
    num_parallel_calls=AUTOTUNE
)

# Validation Dataset

validation_dataset = validation_dataset.map(
    lambda images, labels: (normalization_layer(images), labels),
    num_parallel_calls=AUTOTUNE
)

# Test Dataset

test_dataset = test_dataset.map(
    lambda images, labels: (normalization_layer(images), labels),
    num_parallel_calls=AUTOTUNE
)

# ==========================================================
# Improve Dataset Performance
# ==========================================================

train_dataset = (
    train_dataset
    .cache()
    .shuffle(buffer_size=1000, seed=SEED)
    .prefetch(AUTOTUNE)
)

validation_dataset = (
    validation_dataset
    .cache()
    .prefetch(AUTOTUNE)
)

test_dataset = (
    test_dataset
    .cache()
    .prefetch(AUTOTUNE)
)

In [ ]:
# ==========================================================
# Verify Pixel Range
# ==========================================================

for images, labels in train_dataset.take(1):

    print("Minimum Pixel :", tf.reduce_min(images).numpy())

    print("Maximum Pixel :", tf.reduce_max(images).numpy())

In [23]:
# ==========================================================
# Data Augmentation Layer
# ==========================================================

data_augmentation = tf.keras.Sequential(

    [

        tf.keras.layers.RandomRotation(
            factor=0.05,
            fill_mode="nearest"
        ),

        tf.keras.layers.RandomZoom(
            height_factor=0.05,
            width_factor=0.05,
            fill_mode="nearest"
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.05,
            width_factor=0.05,
            fill_mode="nearest"
        ),

    ],

    name="Data_Augmentation"

)

In [23]:
# ==================================
# CNN Model
# ==================================

from tensorflow.keras import regularizers

model = tf.keras.Sequential([

    # ======================================================
    # Input Layer
    # ======================================================
    tf.keras.layers.Input(shape=(28, 28, 1)),

    # ======================================================
    # Data Augmentation
    # ======================================================
    data_augmentation,

    # ======================================================
    # Block 1
    # ======================================================
    tf.keras.layers.Conv2D(
        32,
        (3,3),
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        32,
        (3,3),
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.25),

    # ======================================================
    # Block 2
    # ======================================================
    tf.keras.layers.Conv2D(
        64,
        (3,3),
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        64,
        (3,3),
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.30),

    # ======================================================
    # Block 3
    # ======================================================
    tf.keras.layers.Conv2D(
        128,
        (3,3),
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        128,
        (3,3),
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.35),

    # ======================================================
    # Block 4
    # ======================================================
    tf.keras.layers.Conv2D(
        256,
        (3,3),
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.40),

    # ======================================================
    # Classification Head
    # ======================================================
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    ),

    tf.keras.layers.Dropout(0.50),

    tf.keras.layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    ),

    tf.keras.layers.Dropout(0.30),

    # ======================================================
    # Output Layer
    # ======================================================
    tf.keras.layers.Dense(
        52,
        activation="softmax"
    )

])

In [24]:
# ==========================================================
# CNN Model
# ==========================================================

from tensorflow.keras import regularizers

model = tf.keras.Sequential([

    # ======================================================
    # Input Layer
    # ======================================================
    tf.keras.layers.Input(shape=(28, 28, 1)),

    # ======================================================
    # Data Augmentation
    # ======================================================
    data_augmentation,

    # ======================================================
    # Block 1
    # ======================================================
    tf.keras.layers.Conv2D(
        32,
        (3, 3),
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        32,
        (3, 3),
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.20),

    # ======================================================
    # Block 2
    # ======================================================
    tf.keras.layers.Conv2D(
        64,
        (3, 3),
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        64,
        (3, 3),
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),

    # ======================================================
    # Block 3
    # ======================================================
    tf.keras.layers.Conv2D(
        128,
        (3, 3),
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        128,
        (3, 3),
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.30),

    # ======================================================
    # Block 4
    # ======================================================
    tf.keras.layers.Conv2D(
        256,
        (3, 3),
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        256,
        (3, 3),
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.35),

    # ======================================================
    # Classification Head
    # ======================================================
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(
        256,
        activation="relu",
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    ),

    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.40),

    tf.keras.layers.Dense(
        128,
        activation="relu",
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    ),

    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),

    # ======================================================
    # Output Layer
    # ======================================================
    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

In [25]:
# ==========================================================
# Compile CNN
# ==========================================================

optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-3
)

model.compile(

    optimizer=optimizer,

    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")
    ]

)

In [ ]:
# ==========================================================
# Display Model Configuration
# ==========================================================

print("=" * 60)

print("Optimizer      :", optimizer.__class__.__name__)

print("Learning Rate  :", float(optimizer.learning_rate.numpy()))

print("Loss Function  : SparseCategoricalCrossentropy")

print("Metric         : SparseCategoricalAccuracy")

print("=" * 60)

In [ ]:
# ==========================================================
# Display Model Architecture
# ==========================================================

# Display model summary
model.summary()

# Save model architecture as an image
tf.keras.utils.plot_model(

    model,

    to_file="cnn_model_architecture.png",

    show_shapes=True,

    show_dtype=True,

    show_layer_names=True,

    expand_nested=True,

    dpi=150

)

print("Model architecture saved as 'cnn_model_architecture.png'")

In [ ]:
# Display

from IPython.display import Image

Image("cnn_model_architecture.png")

In [ ]:
# ==========================================================
# Model Parameters Summary
# ==========================================================

# Total parameters
total_params = model.count_params()

# Trainable parameters
trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in model.trainable_weights
)

# Non-trainable parameters
non_trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in model.non_trainable_weights
)

print("=" * 60)
print("Model Parameters Summary")
print("=" * 60)

print(f"Total Parameters        : {total_params:,}")
print(f"Trainable Parameters    : {trainable_params:,}")
print(f"Non-Trainable Parameters: {non_trainable_params:,}")

print("=" * 60)

In [ ]:
# ==========================================================
# Check Model Status
# ==========================================================

print("=" * 60)

print("Model Built Status :", model.built)

if model.built:
    print("Model is ready for training.")
else:
    print("Model has not been built yet.")

print("=" * 60)

In [ ]:
# ==========================================================
# Import Callbacks
# ==========================================================

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# ==========================================================
# Save Best Model
# ==========================================================

checkpoint = ModelCheckpoint(

    filepath="/kaggle/working/best_character_model.keras",

    monitor="val_accuracy",

    mode="max",

    save_best_only=True,

    save_weights_only=False,

    verbose=1

)
# ==========================================================
# Early Stopping
# ==========================================================

early_stopping = EarlyStopping(

    monitor="val_accuracy",

    mode="max",

    patience=10,

    restore_best_weights=True,

    verbose=1

)

# ==========================================================
# Reduce Learning Rate
# ==========================================================

reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    mode="min",

    factor=0.5,

    patience=3,

    cooldown=1,

    min_delta=1e-4,

    min_lr=1e-6,

    verbose=1

)

# ==========================================================
# Callback List
# ==========================================================

callbacks = [

    checkpoint,

    early_stopping,

    reduce_lr

]

print("=" * 60)
print("Callbacks Configured Successfully")
print("=" * 60)
print("1. ModelCheckpoint")
print("   Monitor        : val_accuracy")
print("   Save Best Only : True")
print()
print("2. EarlyStopping")
print("   Monitor        : val_accuracy")
print("   Patience       : 8")
print("   Restore Best   : True")
print()
print("3. ReduceLROnPlateau")
print("   Monitor        : val_loss")
print("   Factor         : 0.5")
print("   Patience       : 3")
print("   Minimum LR     : 1e-6")
print("=" * 60)

In [ ]:
# ==========================================================
# Train CNN Model
# ==========================================================

EPOCHS = 100

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=EPOCHS,

    callbacks=callbacks,

    verbose=1

)

In [ ]:
# ==========================================================
# Save Best Model
# ==========================================================

MODEL_PATH = "/kaggle/working/best_character_model.keras"

if model.built:
    model.save(MODEL_PATH)

    print("=" * 60)
    print("Best model saved successfully!")
    print(f"Model File : {MODEL_PATH}")
    print("=" * 60)
else:
    print("Model has not been built. Train the model before saving.")

In [ ]:
# ==========================================================
# Training History
# ==========================================================

print("=" * 60)
print("Training History")
print("=" * 60)

print("Available Metrics:")
print(list(history.history.keys()))

print("\nTraining Completed Epochs :", len(history.history["loss"]))

print(f"Final Training Accuracy   : {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy : {history.history['val_accuracy'][-1]:.4f}")

print(f"Final Training Loss       : {history.history['loss'][-1]:.4f}")
print(f"Final Validation Loss     : {history.history['val_loss'][-1]:.4f}")

print(f"Best Validation Accuracy  : {max(history.history['val_accuracy']):.4f}")

print("=" * 60)

In [ ]:
# ==========================================================
# Accuracy Curve
# ==========================================================

plt.figure(figsize=(10,6))

plt.plot(

    history.history["accuracy"],

    marker="o",

    linewidth=2,

    label="Training Accuracy"

)

plt.plot(

    history.history["val_accuracy"],

    marker="s",

    linewidth=2,

    label="Validation Accuracy"

)

best_epoch = np.argmax(history.history["val_accuracy"])

best_accuracy = history.history["val_accuracy"][best_epoch]

plt.scatter(

    best_epoch,

    best_accuracy,

    s=120,

    label=f"Best Val Accuracy = {best_accuracy:.4f}"

)

plt.title("Training vs Validation Accuracy", fontsize=15)

plt.xlabel("Epoch", fontsize=12)

plt.ylabel("Accuracy", fontsize=12)

plt.ylim(0,1)

plt.grid(True)

plt.legend(loc="lower right")

plt.tight_layout()

plt.show()

In [ ]:
# ==========================================================
# Loss Curve
# ==========================================================

plt.figure(figsize=(10,6))

plt.plot(

    history.history["loss"],

    marker="o",

    linewidth=2,

    label="Training Loss"

)

plt.plot(

    history.history["val_loss"],

    marker="s",

    linewidth=2,

    label="Validation Loss"

)

best_epoch = np.argmin(history.history["val_loss"])

best_loss = history.history["val_loss"][best_epoch]

plt.scatter(

    best_epoch,

    best_loss,

    s=120,

    label=f"Lowest Val Loss = {best_loss:.4f}"

)

plt.title("Training vs Validation Loss", fontsize=15)

plt.xlabel("Epoch", fontsize=12)

plt.ylabel("Loss", fontsize=12)

plt.grid(True)

plt.legend(loc="upper right")

plt.tight_layout()

plt.show()

In [ ]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ==========================================================
# Predict Test Dataset
# ==========================================================

# Predict probabilities
predictions = model.predict(
    test_dataset,
    verbose=1
)

# Convert probabilities to class labels
predicted_labels = np.argmax(predictions, axis=1)

# Collect true labels
true_labels = np.concatenate(
    [labels.numpy() for _, labels in test_dataset]
)

print("=" * 60)
print("Prediction Summary")
print("=" * 60)

print("Prediction Shape :", predictions.shape)
print("Predicted Labels :", predicted_labels.shape)
print("True Labels      :", true_labels.shape)

print("=" * 60)

In [ ]:
# ==========================================================
# Classification Report
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

# Calculate Overall Accuracy
test_accuracy = accuracy_score(
    true_labels,
    predicted_labels
)

print("=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

print(f"Overall Test Accuracy : {test_accuracy * 100:.2f}%")

print("\nClassification Report:\n")

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=classes_name,
        digits=4,
        zero_division=0
    )
)

print("=" * 70)

In [ ]:
# ==========================================================
# Confusion Matrix
# ==========================================================

# Compute Confusion Matrix
cm = confusion_matrix(
    y_true=true_labels,
    y_pred=predicted_labels
)

# Create Figure
fig, ax = plt.subplots(figsize=(20, 20))

# Display Confusion Matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes_name
)

disp.plot(
    cmap="Blues",
    include_values=False,      # Recommended for 52 classes
    xticks_rotation=90,
    colorbar=True,
    ax=ax
)

ax.set_title(
    "Confusion Matrix",
    fontsize=18,
    fontweight="bold"
)

ax.set_xlabel("Predicted Class", fontsize=14)
ax.set_ylabel("True Class", fontsize=14)

plt.tight_layout()

plt.show()

In [ ]:
# ==========================================================
# Display Correct Predictions
# ==========================================================

# Collect all test images
test_images = np.concatenate(
    [images.numpy() for images, _ in test_dataset],
    axis=0
)

plt.figure(figsize=(12, 12))

count = 0

for i in range(len(test_images)):

    if predicted_labels[i] == true_labels[i]:

        plt.subplot(4, 4, count + 1)

        plt.imshow(
            test_images[i].squeeze(),
            cmap="gray"
        )

        plt.title(
            f"True : {classes_name[true_labels[i]]}\n"
            f"Pred : {classes_name[predicted_labels[i]]}",
            fontsize=10
        )

        plt.axis("off")

        count += 1

        if count == 16:
            break

plt.tight_layout()

plt.show()

In [ ]:
# ==========================================================
# Display Wrong Predictions
# ==========================================================

# Collect all test images
test_images = np.concatenate(
    [images.numpy() for images, _ in test_dataset],
    axis=0
)

plt.figure(figsize=(12, 12))

count = 0

for i in range(len(test_images)):

    if predicted_labels[i] != true_labels[i]:

        plt.subplot(4, 4, count + 1)

        plt.imshow(
            test_images[i].squeeze(),
            cmap="gray"
        )

        plt.title(
            f"True : {classes_name[true_labels[i]]}\n"
            f"Pred : {classes_name[predicted_labels[i]]}",
            fontsize=10
        )

        plt.axis("off")

        count += 1

        if count == 16:
            break

plt.tight_layout()

plt.show()

In [ ]:
# ==========================================================
# Top-5 Prediction Probabilities
# ==========================================================

NUM_IMAGES = 5

print("=" * 70)
print("Top-5 Prediction Probabilities")
print("=" * 70)

plt.figure(figsize=(15, 3 * NUM_IMAGES))

for i in range(NUM_IMAGES):

    # Display Image
    plt.subplot(NUM_IMAGES, 1, i + 1)

    plt.imshow(test_images[i].squeeze(), cmap="gray")

    plt.title(
        f"True : {classes_name[true_labels[i]]}   |   "
        f"Predicted : {classes_name[predicted_labels[i]]}",
        fontsize=11
    )

    plt.axis("off")

    # Top-5 Predictions
    top5 = np.argsort(predictions[i])[-5:][::-1]

    print(f"\nImage {i + 1}")

    for rank, idx in enumerate(top5, start=1):

        print(
            f"{rank}. "
            f"{classes_name[idx]:<15}"
            f"{predictions[i][idx] * 100:6.2f}%"
        )

    print("-" * 70)

plt.tight_layout()

plt.show()

In [ ]:
# ==========================================================
# Project Performance Summary
# ==========================================================

print("=" * 70)
print("Project Performance Summary")
print("=" * 70)

print(f"Dataset Classes        : {len(classes_name)}")
print(f"Input Image Size       : 28 × 28 × 1")
print(f"Training Epochs        : {len(history.history['loss'])}")
print(f"Final Test Accuracy    : {test_accuracy * 100:.2f}%")
print(f"Saved Model            : best_character_model.keras")

print("=" * 70)
print("Model Training and Evaluation Completed Successfully.")
print("=" * 70)

In [ ]:
# ==========================================================
# Custom Predict All Images in Folder
# ==========================================================

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# ----------------------------------------------------------
# Folder Path
# ----------------------------------------------------------

IMAGE_FOLDER = "/kaggle/input/datasets/rahulkumar1822/images-file"

# ----------------------------------------------------------
# Get All Image Files
# ----------------------------------------------------------

image_paths = sorted(
    glob.glob(os.path.join(IMAGE_FOLDER, "*"))
)

print("=" * 70)
print(f"Total Images Found : {len(image_paths)}")
print("=" * 70)

# ----------------------------------------------------------
# Predict Each Image
# ----------------------------------------------------------

for image_path in image_paths:

    # Load Image
    image = tf.keras.utils.load_img(
        image_path,
        color_mode="grayscale",
        target_size=(28, 28)
    )

    # Convert to Array
    image_array = tf.keras.utils.img_to_array(image)

    # Normalize
    image_array = image_array.astype("float32") / 255.0

    # Add Batch Dimension
    image_array = np.expand_dims(image_array, axis=0)

    # Prediction
    prediction = model.predict(image_array, verbose=0)

    predicted_class = np.argmax(prediction[0])

    confidence = prediction[0][predicted_class]

    # Top-5 Predictions
    top5 = np.argsort(prediction[0])[-5:][::-1]

    # ------------------------------------------------------
    # Display Image
    # ------------------------------------------------------

    plt.figure(figsize=(4,4))

    plt.imshow(image_array[0].squeeze(), cmap="gray")

    plt.title(
        f"Prediction : {classes_name[predicted_class]}\n"
        f"Confidence : {confidence*100:.2f}%"
    )

    plt.axis("off")

    plt.show()

    # ------------------------------------------------------
    # Print Result
    # ------------------------------------------------------

    print("=" * 70)

    print(f"Image Name : {os.path.basename(image_path)}")

    print(f"Predicted Character : {classes_name[predicted_class]}")

    print(f"Confidence : {confidence*100:.2f}%")

    print("\nTop-5 Predictions")

    print("-" * 70)

    for rank, idx in enumerate(top5, start=1):

        print(
            f"{rank}. "
            f"{classes_name[idx]:<15}"
            f"{prediction[0][idx]*100:6.2f}%"
        )

    print("=" * 70)

In [58]:
# End